In [ ]:
# This script calculates heat season characteristics including: onset date, cessation date, and length.
# In the manuscript, it is used for calculating heat season characteristics for the ERA5 data.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import os
import re
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

In [ ]:
# =============================================================================
# Dataset Information: paths to data; years of analysis
# =============================================================================

# Indicate path to daily temperature files. Note, the script will analyze one temperature variable (TMAX or TMIN) at a time.
FILE_PATTERN = ".../T2MDailyMin_*.nc" 
#FILE_PATTERN = ".../T2MDailyMax_*.nc" 


# --- A. FILE LOADING PERIODS  ---
# Load one year EXTRA at the start to catch the previous July-Dec (which is important for dealing with the Southern Hemisphere season lengths)
# Example: If Analysis is 1961-1990, Load 1960-1990
# See Manuscript Methods for details on why this is needed
FILE_P1_START, FILE_P1_END = 1965, 1995  
FILE_P2_START, FILE_P2_END = 1995, 2025  

# --- B. ANALYSIS PERIODS ---
# These are the exact years you want to calculate stats for
ANALYSIS_P1_START, ANALYSIS_P1_END = 1966, 1995
ANALYSIS_P2_START, ANALYSIS_P2_END = 1996, 2025

In [ ]:
# =============================================================================
# 2. DATA LOADING
# =============================================================================
def get_files_for_period(pattern, start_year, end_year):
    """
    Scans directory for files matching the pattern and filters by year.
    """
    all_files = sorted(glob.glob(pattern))
    selected = []
    
    print(f"Scanning {len(all_files)} files for years {start_year}-{end_year}...")
    
    for f in all_files:
        # Robustly extract year from filename
        match = re.search(r'(\d{4})', os.path.basename(f))
        if match:
            year = int(match.group(1))
            if start_year <= year <= end_year:
                selected.append(f)
    
    if not selected:
        print(f"  WARNING: No files found for {start_year}-{end_year}")
        
    return sorted(selected)

def load_data(file_list):
    """Loads files lazily with Dask."""
    if not file_list: return None
    
    # Open with Dask
    ds = xr.open_mfdataset(file_list, combine='by_coords', parallel=True, chunks={'time': 365})
    
    # Auto-detect variable
    var_name = None
    for v in ['t2m', 'mx2t', 'tasmax', 'var167', 'VAR_2T']: 
        if v in ds:
            var_name = v
            break
    if var_name is None: var_name = list(ds.data_vars)[0]
        
    data = ds[var_name]

    # Standardize Coords
    if 'latitude' in data.coords: data = data.rename({'latitude': 'lat'})
    if 'longitude' in data.coords: data = data.rename({'longitude': 'lon'})
    if 'expver' in data.dims: data = data.reduce(np.nansum, dim='expver')

    # Convert K to C
    if data.isel(time=0).max().compute().item() > 100:
        print(f"  -> Converting {var_name} from Kelvin to Celsius...")
        data = data - 273.15
        
    return data

def calculate_threshold(data, start_year, end_year, percentile=0.95):
    """
    Calculates the threshold using ONLY the analysis years.
    Ignores any buffer years (like 1960) loaded for SH processing.
    """
    print(f"Calculating {int(percentile*100)}th percentile threshold for {start_year}-{end_year}...")
    
    # STRICT SLICING: Only use the analysis period
    # We slice the provided 'data' (which might be 1960-1990) to just 1961-1990
    analysis_slice = data.sel(time=slice(str(start_year), str(end_year)))
    
    # Compute Quantile
    threshold = analysis_slice.quantile(percentile, dim='time').compute()
    return threshold



In [ ]:
# 1. Identify Files (catch buffer year)
files_p1 = get_files_for_period(FILE_PATTERN, FILE_P1_START, FILE_P1_END)
files_p2 = get_files_for_period(FILE_PATTERN, FILE_P2_START, FILE_P2_END)
        
# 2. Load Data
ds_p1 = load_data(files_p1)
ds_p2 = load_data(files_p2)

In [ ]:
# use this for the bimodal calculation and splitting seasons for bimodal analysis
all_files = sorted(files_p1 + files_p2)
ds_all = load_data(all_files)

In [ ]:
# 3. Calculate Threshold (Baseline Analysis Years)
# We pass 1966-1995 here, ignoring the 1965 buffer data
thresh_p1 = calculate_threshold(ds_p1, ANALYSIS_P1_START, ANALYSIS_P1_END, percentile=0.97)

In [ ]:
# Calculate what the threshold is for second period for comparison purposes only.
thresh_p2 = calculate_threshold(ds_p2, ANALYSIS_P2_START, ANALYSIS_P2_END, percentile=0.97)

In [ ]:
diff_threshold = thresh_p2 - thresh_p1

In [ ]:
# check on data structure
diff_threshold

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
import datetime

def map_tropical_bimodality(raw_data, threshold_map, 
                            min_peak_distance=90,        
                            prominence_fraction=0.10, 
                            max_valley_ratio=0.30,
                            min_years_per_peak=10,        
                            peak_window_days=30):        
    """
    Creates maps of extreme heat seasonality, including the number of modes,
    the shallow valley DOY (for S1/S2 splits), and the deep valley month
    (for explicitly locking the calendar boundary).
    """
    print(f"Starting Bimodality & Valley Mapping...")
    print(f"  Parameters: Distance > {min_peak_distance}d, Prominence > {prominence_fraction*100}%, Valley Depth < {max_valley_ratio*100}%")
    
    lat_bound = 23.5
    if raw_data.lat[0] > raw_data.lat[-1]:
        trop_slice = slice(lat_bound, -lat_bound)
    else:
        trop_slice = slice(-lat_bound, lat_bound)
        
    data_trop = raw_data.sel(lat=trop_slice)
    thresh_trop = threshold_map.sel(lat=trop_slice)
    
    print("  Calculating extreme days mask (this may take a moment)...")
    is_extreme = (data_trop > thresh_trop).compute().values
    
    lats = data_trop.lat.values
    lons = data_trop.lon.values
    doys = data_trop.time.dt.dayofyear.values
    years = data_trop.time.dt.year.values 
    
    peak_count_map = np.full((len(lats), len(lons)), np.nan)
    valley_doy_map = np.full((len(lats), len(lons)), np.nan) 
    deep_valley_month_map = np.full((len(lats), len(lons)), np.nan) # NEW MAP
    
    x_eval = np.arange(1, 366)
    
    for i, lat in enumerate(lats):
        if i % 10 == 0: 
            print(f"    Processing Latitude {i}/{len(lats)} ({lat:.2f}°)...", end='\r')
            
        for j, lon in enumerate(lons):
            pixel_extremes = is_extreme[:, i, j]
            if not np.any(pixel_extremes):
                continue
                
            pixel_doys = doys[pixel_extremes]
            pixel_years = years[pixel_extremes] 
            
            if len(pixel_doys) < 30:
                peak_count_map[i, j] = 1 
                continue
                
            doys_extended = np.concatenate([pixel_doys - 365, pixel_doys, pixel_doys + 365])
            
            try:
                kde = gaussian_kde(doys_extended, bw_method=0.05)
                density = kde.evaluate(x_eval)
                
                max_density = np.max(density)
                min_prominence = max_density * prominence_fraction
                peaks, _ = find_peaks(density, distance=min_peak_distance, prominence=min_prominence)
                
                # --- PHASE 1: DISTANCE & VALLEY CHECK ---
                valid_peaks = list(peaks)
                merged = True
                
                while merged and len(valid_peaks) > 1:
                    merged = False
                    valid_peaks.sort() 
                    
                    for k in range(len(valid_peaks)):
                        p1 = valid_peaks[k]
                        p2 = valid_peaks[(k + 1) % len(valid_peaks)]
                        
                        dist = min((p2 - p1) % 365, (p1 - p2) % 365)
                        if dist < min_peak_distance:
                            drop_p = p1 if density[p1] < density[p2] else p2
                            valid_peaks.remove(drop_p)
                            merged = True
                            break 
                        
                        if p1 < p2:
                            valley = np.min(density[p1:p2])
                        else:
                            valley = min(np.min(density[p1:]), np.min(density[:p2]))
                            
                        smaller_peak = min(density[p1], density[p2])
                        
                        if valley > smaller_peak * max_valley_ratio:
                            drop_p = p1 if density[p1] < density[p2] else p2
                            valid_peaks.remove(drop_p)
                            merged = True
                            break 

                # --- PHASE 2: TEMPORAL CONSISTENCY CHECK ---
                robust_peaks = []
                for p in valid_peaks:
                    p_doy = x_eval[p]
                    dist = np.minimum((pixel_doys - p_doy) % 365, (p_doy - pixel_doys) % 365)
                    contributing_years = np.unique(pixel_years[dist <= peak_window_days])
                    if len(contributing_years) >= min_years_per_peak:
                        robust_peaks.append(p)
                        
                valid_peaks = robust_peaks
                num_peaks = max(1, len(valid_peaks))
                peak_count_map[i, j] = num_peaks

                # --- PHASE 3: IDENTIFY SHALLOW AND DEEP VALLEYS ---
                if num_peaks >= 2:
                    valid_peaks.sort()
                    p1 = valid_peaks[0]
                    p2 = valid_peaks[1] 
                    
                    v1_idx = np.argmin(density[p1:p2]) + p1
                    v1_density = density[v1_idx]
                    
                    idx_after = np.argmin(density[p2:]) + p2
                    idx_before = np.argmin(density[:p1])
                    
                    if density[idx_after] < density[idx_before]:
                        v2_idx = idx_after
                    else:
                        v2_idx = idx_before
                    v2_density = density[v2_idx]
                    
                    # Explicitly define Shallow vs. Deep
                    if v1_density > v2_density:
                        shallow_idx = v1_idx
                        deep_idx = v2_idx
                    else:
                        shallow_idx = v2_idx
                        deep_idx = v1_idx
                        
                    # 1. Assign the Shallow Valley DOY for the S1/S2 Split
                    valley_doy_map[i, j] = x_eval[shallow_idx]
                    
                    # 2. Assign the Deep Valley Month for the Calendar Split
                    deep_doy = int(x_eval[deep_idx])
                    # Using a non-leap year dummy to reliably convert DOY to Month (1-12)
                    deep_month = (datetime.datetime(2001, 1, 1) + datetime.timedelta(days=deep_doy - 1)).month
                    deep_valley_month_map[i, j] = deep_month
                
            except Exception as e:
                continue

    print(f"\n  Done! Map generation complete.        ")
    
    ds_out = xr.Dataset(
        {
            'season_modes': (['lat', 'lon'], peak_count_map),
            'valley_doy': (['lat', 'lon'], valley_doy_map),
            'deep_valley_month': (['lat', 'lon'], deep_valley_month_map) 
        },
        coords={'lat': lats, 'lon': lons}
    )
    
    return ds_out

In [ ]:
bimodal_results_all = map_tropical_bimodality(ds_p1, thresh_p1) # use this if only passing the ref period to the bimodal finder

In [ ]:
modes_map_all = bimodal_results_all['season_modes']
valley_map_all = bimodal_results_all['valley_doy']

In [ ]:
# =============================================================================
# 3. METRICS CALCULATION 
# =============================================================================
def get_year_metrics(data_slice, threshold, year, is_southern):
    """
    Calculates start, end, and duration.
    It wraps start/end dates back to 1-365 range for plotting.
    """
    doy = data_slice.time.dt.dayofyear
    
    # 1. Determine Year Length (Critical for correct Modulo math)
    # If this is a leap year, we need to modulo by 366, otherwise 365.
    days_in_year = 366 if pd.Timestamp(f"{year}-12-31").is_leap_year else 365

    # 2. Shift Southern Hemisphere
    if is_southern:
        # Shift Jan-Jun (DOY < 183) to follow Dec of previous year.
        # We add days_in_year so the math works for duration.
        adjusted_doy = xr.where(doy < 183, doy + days_in_year, doy)
    else:
        adjusted_doy = doy

    # 3. Identify Extreme Days
    is_extreme = data_slice >= threshold
    extreme_days = adjusted_doy.where(is_extreme)

    # 4. Calculate Monotonic Metrics
    # These might exceed 365 (e.g., Day 367)
    f = extreme_days.min(dim='time')
    l = extreme_days.max(dim='time')
    
    # 5. Calculate Duration (Using Monotonic values)
    length = (l - f) + 1
    length = length.fillna(0) 

    # 6. NORMALIZE START/END DATES
    # We wrap the values back to the 1-365/366 calendar range.
    # Logic: (Day - 1) % days_in_year + 1
    # Example (Leap Year): Day 367 -> (366) % 366 + 1 -> 1 (Jan 1st)
    f_clean = (f - 1) % days_in_year + 1
    l_clean = (l - 1) % days_in_year + 1

    return f_clean, l_clean, length


def calculate_period_stats(data, threshold, start_year, end_year):
    """
    Loops through analysis years and computes metrics for NH and SH.
    """
    years = np.arange(start_year, end_year + 1)
    
    nh_first, nh_last, nh_len = [], [], []
    sh_first, sh_last, sh_len = [], [], []

    # Separate Thresholds
    nh_thresh = threshold.where(threshold.lat >= 0, drop=True)
    sh_thresh = threshold.where(threshold.lat < 0, drop=True)

    print(f"  Computing metrics for analysis period {start_year}-{end_year}...")
    
    for year in years:
        # --- NH Processing (Jan-Dec) ---
        try:
            nh_data = data.sel(time=str(year)).where(data.lat >= 0, drop=True)
            if nh_data.time.size > 0:
                f, l, d = get_year_metrics(nh_data, nh_thresh, year, is_southern=False)
                nh_first.append(f.compute())
                nh_last.append(l.compute())
                nh_len.append(d.compute())
        except KeyError: pass

        # --- SH Processing (July Year-1 to June Year) ---
        try:
            prev_year = year - 1
            t_start, t_end = f"{prev_year}-07-01", f"{year}-06-30"
            
            # Note: We assume 'data' contains the buffer year (prev_year)
            sh_data = data.sel(time=slice(t_start, t_end)).where(data.lat < 0, drop=True)
            
            if sh_data.time.size > 300: 
                f, l, d = get_year_metrics(sh_data, sh_thresh, year, is_southern=True)
                sh_first.append(f.compute())
                sh_last.append(l.compute())
                sh_len.append(d.compute())
        except KeyError: pass
        
        print(f"    Processed {year}", end='\r')
    print("")

    return (nh_first, nh_last, nh_len, sh_first, sh_last, sh_len)



# =============================================================================
# TROPICAL SLICING
# =============================================================================
def get_tropical_subset(data, lat_bound=23.5):
    """
    Safely slices the dataset to the tropics, handling N->S or S->N sorting.
    """
    lat_values = data.lat.values
    if lat_values[0] > lat_values[-1]:
        # Decreasing (90 -> -90)
        return data.sel(lat=slice(lat_bound, -lat_bound))
    else:
        # Increasing (-90 -> 90)
        return data.sel(lat=slice(-lat_bound, lat_bound))


# =============================================================================
# DISCOVERY: START MONTH (Deepest Minimum Window)
# =============================================================================
def get_tropical_start_month(trop_data, trop_threshold):
    """
    Determines the optimal start month for tropical pixels.
    Uses a cyclically wrapped rolling sum to find the deepest,
    longest period of inactivity (the 'coolest' part of the year),
    placing the calendar boundary safely in the middle of it.
    """
    print("  Determining start months (Deepest Minimum Window)...")
    
    # 1. Climatology: Count of extreme days per month
    is_extreme = trop_data > trop_threshold
    monthly_clim = is_extreme.groupby('time.month').sum(dim='time')
    
    # 2. Cyclical Padding: Concatenate 3 times to perfectly handle Dec -> Jan wrap-around
    # Shape becomes 36 months: [Jan..Dec, Jan..Dec, Jan..Dec]
    clim_padded = xr.concat([monthly_clim, monthly_clim, monthly_clim], dim='month')
    
    # 3. Rolling Window: A 5-month window sweeps across the padded data
    # The center of the longest streak of 0s will produce the lowest rolling sum
    rolling_sum = clim_padded.rolling(month=5, center=True).sum()
    
    # 4. Extract the true middle 12 months (which now contain fully wrapped sums)
    middle_rolling = rolling_sum.isel(month=slice(12, 24))
    
    # 5. Find the Minimum: .argmin() returns the positional index (0-11)
    # Adding 1 converts it to the standard 1-12 calendar month
    best_idx = middle_rolling.argmin(dim='month')
    start_month = best_idx + 1
    
    return start_month.fillna(1).astype(int).drop_vars('time', errors='ignore')





def calculate_tropical_metrics(data, threshold, start_year, end_year, 
                               ref_start_month_map=None,
                               bimodal_modes_map=None,   
                               bimodal_valley_map=None,
                               bimodal_start_month_map=None): 
    """
    Calculates flexible metrics only for the tropics.
    Now overrides the rolling-sum map with the KDE deep valley map for bimodal pixels.
    """
    print(f"Starting Tropical Analysis ({start_year}-{end_year})...")
    
    data_trop = get_tropical_subset(data)
    thresh_trop = get_tropical_subset(threshold)
    
    # ==========================================================
    # --- MERGE START MONTH MAPS ---
    # ==========================================================
    if ref_start_month_map is not None:
        start_month_map = get_tropical_subset(ref_start_month_map)
    else:
        # 1. Base map from 5-month rolling sum (Unimodal pixels use this)
        base_start_month = get_tropical_start_month(data_trop, thresh_trop)
        
        # 2. Merge logic: If bimodal, override with the deep valley KDE map
        if bimodal_modes_map is not None and bimodal_start_month_map is not None:
            bimodal_override = get_tropical_subset(bimodal_start_month_map)
            modes_map_trop = get_tropical_subset(bimodal_modes_map)
            
            # Where modes >= 2, use bimodal_override. Otherwise use base_start_month
            start_month_map = xr.where(modes_map_trop >= 2, bimodal_override, base_start_month)
            # Ensure it remains a clean integer map
            start_month_map = start_month_map.fillna(1).astype(int)
        else:
            start_month_map = base_start_month
            
    # ==========================================================
    # --- BIMODAL MAPS ASSIGNMENT ---
    # ==========================================================
    if bimodal_modes_map is not None and bimodal_valley_map is not None:
        modes_map = get_tropical_subset(bimodal_modes_map)
        valley_map = get_tropical_subset(bimodal_valley_map)
    else:
        modes_map = xr.zeros_like(start_month_map) + 1
        valley_map = xr.full_like(start_month_map, np.nan)
    
    results_list = []

    for m in range(1, 13):
        mask = (start_month_map == m).compute()
        if not mask.any(): continue
            
        print(f"  Computing Start Month {m}...", end='\r')
        
        group_data = data_trop.where(mask, drop=True)
        group_thresh = thresh_trop.where(mask, drop=True)
        group_modes = modes_map.where(mask, drop=True)
        group_valley = valley_map.where(mask, drop=True)
        
        years = np.arange(start_year, end_year + 1)
        
        for yr in years:
            days_in_year = 366 if pd.Timestamp(f"{yr}-12-31").is_leap_year else 365

            if m == 1:
                t_start, t_end = f"{yr}-01-01", f"{yr}-12-31"
            else:
                t_start = f"{yr-1}-{m:02d}-01"
                ts_end = pd.Timestamp(f"{yr}-{m}-01") - pd.Timedelta(days=1)
                t_end = ts_end.strftime("%Y-%m-%d")

            try:
                yr_slice = group_data.sel(time=slice(t_start, t_end))
            except KeyError: continue

            if yr_slice.time.size < 300: continue

            raw_doy = yr_slice.time.dt.dayofyear
            is_extreme = yr_slice > group_thresh
            extreme_days = raw_doy.where(is_extreme)
            
            if m != 1:
                approx_start_doy = (m - 1) * 30 
                shifted_extremes = xr.where(extreme_days < approx_start_doy, 
                                            extreme_days + days_in_year, 
                                            extreme_days)
                shifted_valley = xr.where(group_valley < approx_start_doy,
                                          group_valley + days_in_year,
                                          group_valley)
            else:
                shifted_extremes = extreme_days
                shifted_valley = group_valley
            
            # --- Unimodal Boundaries ---
            f_uni = shifted_extremes.min(dim='time')
            l_uni = shifted_extremes.max(dim='time')
            len_uni = (l_uni - f_uni) + 1
            
            # --- Bimodal Boundaries ---
            s1_days = shifted_extremes.where(shifted_extremes <= shifted_valley)
            s2_days = shifted_extremes.where(shifted_extremes > shifted_valley)
            
            s1_f = s1_days.min(dim='time')
            s1_l = s1_days.max(dim='time')
            s2_f = s2_days.min(dim='time')
            s2_l = s2_days.max(dim='time')
            
            len1 = (s1_l - s1_f) + 1
            len2 = (s2_l - s2_f) + 1
            len_bimodal = len1.fillna(0) + len2.fillna(0)
            
            # --- Merge Based on Pixel Type ---
            is_bimodal_pixel = group_modes >= 2
            
            final_len = xr.where(is_bimodal_pixel, len_bimodal, len_uni)
            final_len = final_len.fillna(0).drop_vars('time', errors='ignore').compute()
            
            # Convert ALL boundaries back to standard DOY
            def clean_doy(da):
                return (da.compute() - 1) % days_in_year + 1
                
            s1_start = xr.where(is_bimodal_pixel, clean_doy(s1_f), clean_doy(f_uni))
            s1_end   = xr.where(is_bimodal_pixel, clean_doy(s1_l), clean_doy(l_uni))
            
            # Unimodal pixels get NaNs for Season 2
            s2_start = xr.where(is_bimodal_pixel, clean_doy(s2_f), np.nan)
            s2_end   = xr.where(is_bimodal_pixel, clean_doy(s2_l), np.nan)
            
            # Format for output
            def expand(da):
                return da.reindex_like(start_month_map).expand_dims(year=[yr])
                
            results_list.append({
                'year': yr, 'len': expand(final_len), 
                's1_start': expand(s1_start), 's1_end': expand(s1_end),
                's2_start': expand(s2_start), 's2_end': expand(s2_end)
            })

    print("")
    print("  Aggregating final tropical results...")
    
    def combine_list(key):
        return xr.concat([item[key] for item in results_list], dim='temp').groupby('year').mean(dim='temp')

    final_len = xr.concat([item['len'] for item in results_list], dim='temp').groupby('year').sum(dim='temp')
    final_s1_start = combine_list('s1_start')
    final_s1_end = combine_list('s1_end')
    final_s2_start = combine_list('s2_start')
    final_s2_end = combine_list('s2_end')
    
    return final_len, final_s1_start, final_s1_end, final_s2_start, final_s2_end, start_month_map


In [ ]:
def run_unified_global_analysis(data, threshold, start_year, end_year, output_file="Global_Extreme_Heat_Unified.nc", 
                                ref_start_month_map=None, bimodal_modes_map=None, 
                                bimodal_valley_map=None, bimodal_start_month_map=None): 
    
    print(f"=== STARTING UNIFIED GLOBAL ANALYSIS ({start_year}-{end_year}) ===")

    print("\n>>> Phase 1: Tropical Calculation (+/- 23.5 Lat)")
    trop_len, trop_s1_start, trop_s1_end, trop_s2_start, trop_s2_end, trop_map = calculate_tropical_metrics(
        data, threshold, start_year, end_year, 
        ref_start_month_map=ref_start_month_map,
        bimodal_modes_map=bimodal_modes_map,   
        bimodal_valley_map=bimodal_valley_map,
        bimodal_start_month_map=bimodal_start_month_map 
    )

    print("\n>>> Phase 2: Extra-Tropical Calculation (NH/SH)")
    et_stats = calculate_period_stats(data, threshold, start_year, end_year)
    
    nh_start_list, nh_end_list, nh_len_list = et_stats[0], et_stats[1], et_stats[2]
    sh_start_list, sh_end_list, sh_len_list = et_stats[3], et_stats[4], et_stats[5]

    def list_to_3d(nh_list, sh_list):
        nh_da = xr.concat(nh_list, dim='year')
        sh_da = xr.concat(sh_list, dim='year')
        return xr.concat([nh_da, sh_da], dim='lat').sortby('lat')

    et_len = list_to_3d(nh_len_list, sh_len_list)
    et_s1_start = list_to_3d(nh_start_list, sh_start_list)
    et_s1_end = list_to_3d(nh_end_list, sh_end_list)
    
    # Extra-tropics are unimodal, so Season 2 is just NaNs
    et_s2_start = xr.full_like(et_s1_start, np.nan)
    et_s2_end = xr.full_like(et_s1_end, np.nan)

    print("\n>>> Phase 3: Merging & Saving")
    
    ds_out = xr.Dataset({
        'season_length': et_len,
        's1_start': et_s1_start,
        's1_end': et_s1_end,
        's2_start': et_s2_start,
        's2_end': et_s2_end
    })
    
    # Reindex and merge tropics into the global dataset
    lat_bound = 23.5
    trop_slice = slice(-lat_bound, lat_bound) if ds_out.lat[0] < ds_out.lat[-1] else slice(lat_bound, -lat_bound)

    def merge_tropics(global_var, trop_var):
        aligned = trop_var.reindex(lat=ds_out.lat, method='nearest', tolerance=0.01)
        ds_out[global_var].loc[{'lat': trop_slice}] = aligned.sel(lat=trop_slice)

    merge_tropics('season_length', trop_len)
    merge_tropics('s1_start', trop_s1_start)
    merge_tropics('s1_end', trop_s1_end)
    merge_tropics('s2_start', trop_s2_start)
    merge_tropics('s2_end', trop_s2_end)

    encoding = {var: {'_FillValue': np.nan, 'dtype': 'float64', 'zlib': True} for var in ds_out.data_vars}
    ds_out.to_netcdf(output_file, encoding=encoding)
    print(f"Done! Unified results saved to {output_file}")

    stats = {'ds_merged': ds_out, 'trop_map': trop_map}
    return stats



In [ ]:
# ==========================================
# 1. RUN THE UNIFIED GLOBAL ANALYSIS
# ==========================================

# 1. Extract all three bimodal maps from the function
bimodal_results_all = map_tropical_bimodality(ds_p1, thresh_p1) 

modes_map_all = bimodal_results_all['season_modes']
valley_map_all = bimodal_results_all['valley_doy']
deep_month_map_all = bimodal_results_all['deep_valley_month'] 

# 2. Run Baseline Analysis
stats_p1 = run_unified_global_analysis(
    ds_p1, thresh_p1, ANALYSIS_P1_START, ANALYSIS_P1_END, "ADJUSTED_ERA5_Global_1966_1995_TMIN.nc",
    bimodal_modes_map=modes_map_all,
    bimodal_valley_map=valley_map_all,
    bimodal_start_month_map=deep_month_map_all # <--- PASS IT IN
)

# 3. Save the strictly locked, bimodal-aware map for P2 to inherit
trop_map_p1 = stats_p1['trop_map']

# 4. Run Future Analysis (P2)
stats_p2 = run_unified_global_analysis(
    ds_p2, thresh_p1, ANALYSIS_P2_START, ANALYSIS_P2_END, "ADJUSTED_ERA5_Global_1996_2025_TMIN.nc",
    ref_start_month_map=trop_map_p1, # Inheriting the locked map from P1
    bimodal_modes_map=modes_map_all,
    bimodal_valley_map=valley_map_all,
    bimodal_start_month_map=deep_month_map_all # Also strictly inherited
)

In [ ]:
ds1 = stats_p1['ds_merged']
ds2 = stats_p2['ds_merged']

ds1['ref_trop_start_month'] = trop_map_p1
ds2['ref_trop_start_month'] = trop_map_p1

In [ ]:
ds1

In [ ]:
ds2

In [ ]:
# OUTPUT THE DS DATASETS CONTAINING THE HEAT SEASON CHARACTERISTICS TO BRING THEM INTO OTHER SCRIPTS

In [ ]:
# 1. Ensure the dataset is "clean" for NetCDF (remove any leftover dask/time coordinates)
ds2_out_ref_method = ds2.drop_vars(['quantile', 'time'], errors='ignore')
ds1_out_ref_method = ds1.drop_vars(['quantile', 'time'], errors='ignore')

In [ ]:
# 1. Ensure the dataset is "clean" for NetCDF
ds2_out_ref_method = ds2.drop_vars(['quantile', 'time'], errors='ignore')
ds1_out_ref_method = ds1.drop_vars(['quantile', 'time'], errors='ignore')


# Clear the source encoding so xarray forgets all about TMAX
ds2_out_ref_method.encoding.pop('source', None)
ds1_out_ref_method.encoding.pop('source', None)

In [ ]:
import os

# Set the location for the new netcdf files to be placed
OUTPUT_DIR = "/.../"

# Explicitly join the directory path to the filename
output_filename_ref2 = os.path.join(OUTPUT_DIR, "ERA5_TMIN_1996-2025_HeatSeasonCharacteristics.nc")
output_filename_ref1 = os.path.join(OUTPUT_DIR, "ERA5_TMIN_1966-1995_HeatSeasonCharacteristics.nc")

# 3. Save with compression to save disk space
print("Saving files to disk (this may take a minute)...")
encoding2 = {var: {'zlib': True, 'complevel': 4} for var in ds2_out_ref_method.data_vars}
encoding1 = {var: {'zlib': True, 'complevel': 4} for var in ds1_out_ref_method.data_vars}

ds2_out_ref_method.to_netcdf(output_filename_ref2, encoding=encoding2)
ds1_out_ref_method.to_netcdf(output_filename_ref1, encoding=encoding1)

print(f"\nSUCCESS! Files explicitly saved to:")
print(f" -> {output_filename_ref1}")
print(f" -> {output_filename_ref2}")